In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import mediapipe as mp

In [4]:
df = pd.read_excel(r"D:\DEMO\csv\log_temp_files\new_log_seq-26-06-02-11-39-12.xlsx", header = None)
df.head()

,0,1,2,3,4,5,6,7,8,9,...,630,631,632,633,634,635,636,637,638,639
0,27.468010,27.485804,27.502846,27.494570,27.476080,27.458681,27.489063,27.428171,27.403502,27.409248,...,26.137840,26.098269,26.119713,26.165754,26.181198,26.099501,26.068182,26.036011,26.044161,26.092251
1,27.480080,27.522282,27.510437,27.487123,27.429934,27.418192,27.466539,27.474752,27.449600,27.461546,...,26.166656,26.145021,26.179628,26.194103,26.138901,26.179710,26.113485,26.056637,26.119541,26.190199
2,27.494368,27.531025,27.518110,27.490345,27.452333,27.497810,27.461918,27.452513,27.492413,27.530687,...,26.184830,26.135658,26.149971,26.110195,26.086542,26.109350,26.157639,26.172556,26.181509,26.163877
3,27.429655,27.470875,27.510485,27.522224,27.500628,27.471731,27.433294,27.434368,27.448563,27.512548,...,26.175152,26.123774,26.073357,26.044485,26.031532,26.069529,26.157166,26.163395,26.130669,26.134899
4,27.411856,27.434750,27.486980,27.534349,27.519390,27.467754,27.431625,27.433968,27.462521,27.479336,...,26.170145,26.130573,26.073141,26.025442,26.028355,26.067633,26.108725,26.056599,26.033947,26.066442


In [5]:
df.shape

(480, 640)

In [7]:
arr = df.to_numpy()
arr.shape

(480, 640)

## Visualizing the temperature file

In [1]:
import pandas as pd
import numpy as np
import cv2

df = pd.read_excel(r"D:\DEMO\csv\log_temp_files\new_log_seq-26-06-02-11-39-12.xlsx", header = None)

arr = df.to_numpy(dtype=np.float32)

print(arr.shape)

(480, 640)


In [12]:
img = cv2.normalize(
    arr,
    None,
    0,
    255,
    cv2.NORM_MINMAX
)

img = img.astype(np.uint8)

In [15]:
cv2.imshow("Thermal Gray", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [14]:
colored = cv2.applyColorMap(
    img,
    cv2.COLORMAP_JET
)

cv2.imshow("Thermal Color", colored)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [16]:
import cv2
import numpy as np
# ============================================================
# CLAHE (Create Once)
# ============================================================
clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)

def get_transformed_image(grey_frame):
    """
    Applies contrast enhancement pipeline to the input grayscale frame and returns the enhanced greyscaled frame.
    """

    # ========================================================
    # CONTRAST ENHANCEMENT PIPELINE
    # ========================================================

    # Stretch contrast
    stretched = cv2.normalize(
        grey_frame,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    # CLAHE
    enhanced = clahe.apply(stretched)

    # Gamma correction
    gamma_corrected = (
        np.power(enhanced / 255.0, 0.6) * 255
    ).astype(np.uint8)

    # Unsharp mask
    blurred = cv2.GaussianBlur(
        gamma_corrected,
        (0, 0),
        3
    )

    # sharpened = cv2.addWeighted(
    #     gamma_corrected,
    #     1.5,
    #     blurred,
    #     -0.5,
    #     0
    # )

    sharpened_grey_frame = cv2.addWeighted(                        # sharpened the edges
    gamma_corrected,
    2,
    blurred,
    -1.5,
    0
   )
    
    return sharpened_grey_frame                                  # returning the enhanced greyscaled frame


In [17]:
transformed_img = get_transformed_image(img)

In [18]:
cv2.imshow("TRANSFORMED window", transformed_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [19]:
def get_eyes_coordinates(frame, grey, face_landmarks):
            
            import mediapipe as mp
            import cv2

            # Inner eye corner landmarks
            LEFT_INNER_EYE = 133                                      # mediapipe landmark for left eye inner corner
            RIGHT_INNER_EYE = 362                                     # mediapipe landmark for right eye inner corner
                               
            PERCENTAGE_PIXEL_TO_KEEP = 0.80
            h, w, _ = frame.shape                           # mediapipe landmark for right eye inner corner

            # LEFT INNER EYE   --->   lx is the x-coordinate of inner corner for left eye and ly is the y coordinate of inner corner of left eye
            left_point = face_landmarks.landmark[LEFT_INNER_EYE]                     
            lx = int(left_point.x * w)
            ly = int(left_point.y * h)

            # RIGHT INNER EYE  --->  rx is the x-coordinate of inner corner for right eye and ry is the y coordinate of inner corner of right eye
            right_point = face_landmarks.landmark[RIGHT_INNER_EYE]
            rx = int(right_point.x * w)
            ry = int(right_point.y * h)
 
            # For LEFT INNER EYE
            top_left_coords = (lx, ly-10)
            bottom_right_coords = (lx+20, ly+10)
  
            # For RIGHT INNER EYE   
            top_right_coords = (rx, ry-10)
            bottom_left_coords = (rx-20, ry+10)

            # CV2.CIRCLE is modifyng the image array (Just changing, not DISPLAYING)
            # displaying the top left and bottom right corner of the box around inner eye corners LEFT EYE
            cv2.circle(frame, top_left_coords, 2, (0, 255, 0), -5)    #dot above left inner eye corner
            cv2.circle(frame, bottom_right_coords, 2, (0, 255, 0), -5)    #dot right to left inner eye corner

            # displaying the top left and bottom right corner of the box around inner eye corners RIGHT EYE
            cv2.circle(frame, top_right_coords, 2, (0, 255, 0), -5)    #dot above left inner eye corner
            cv2.circle(frame, bottom_left_coords, 2, (0, 255, 0), -5)    #dot right to left inner eye corner

            cv2.rectangle(frame, top_left_coords, bottom_right_coords, (255, 0, 0), 2)    #rectangle around left inner eye corner
            cv2.rectangle(frame, top_right_coords, bottom_left_coords, (255, 0, 0), 2)    #rectangle around right inner eye corner


            return top_left_coords, bottom_right_coords, top_right_coords, bottom_left_coords


In [27]:
import cv2
import mediapipe as mp

# img = cv2.imread("face.jpg")
img_rgb = cv2.cvtColor(transformed_img, cv2.COLOR_BGR2RGB)

mp_face_mesh = mp.solutions.face_mesh

 # Convert grayscale -> BGR for MediaPipe   # Gibing transformed/enhanced frame to mediapipe for better detection of landmarks.
rgb = cv2.cvtColor(
        transformed_img,
        cv2.COLOR_GRAY2BGR
    )


# Some eye landmarks
LEFT_EYE = [133]
RIGHT_EYE = [362]

with mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    refine_landmarks=True
) as face_mesh:

    results = face_mesh.process(img_rgb)

    if results.multi_face_landmarks:

        h, w, _ = rgb.shape

        for face_landmarks in results.multi_face_landmarks:

            for idx in LEFT_EYE + RIGHT_EYE:

                lm = face_landmarks.landmark[idx]

                x = int(lm.x * w)
                y = int(lm.y * h)

                cv2.circle(transformed_img, (x, y), 3, (0, 0, 255), -1)

cv2.imshow("Eyes", transformed_img)
cv2.waitKey(0)
cv2.destroyAllWindows()